# Семинар 8. Последовательности, RNN, GRU, LSTM и CRNN

Основная часть следует Chapter 8 D. V. Godoy в авторском порядке:
данные → RNN → обучение и визуализация → GRU → LSTM → переменные длины → Conv1d → общая модель.
CRNN, CTC и материал прошлого года находятся после этой базы. STN завершает приложение.
Материал расширен без ограничения доклада 90 минут.

Короткие ручные расчёты и диагностические графики выполняются на CPU явно.
Обучение использует выбранное устройство MPS → CUDA → CPU, без скрытого fallback.
Для полного CPU-запуска задайте `AI_MASTERS_DEVICE=cpu` перед шапкой.
OCR-архив доступен по постоянной ссылке; обучение моделей Годоя не зависит от него.

## Содержание

1. [Последовательности и данные квадрата](#section-1)
2. [RNN: от состояния к ячейке и слою](#section-2)
3. [Многослойная RNN](#section-3)
4. [Двунаправленная RNN](#section-4)
5. [Square Model: обучение RNN](#section-5)
6. [Что происходит со входами и состояниями](#section-6)
7. [GRU: гейты и ручной шаг](#section-7)
8. [Square Model II: обучение GRU](#section-8)
9. [Визуализация GRU](#section-9)
10. [LSTM: два состояния и ручной шаг](#section-10)
11. [Square Model III: обучение LSTM](#section-11)
12. [Сравнение состояний RNN, GRU и LSTM](#section-12)
13. [Последовательности переменной длины](#section-13)
14. [Dataset, collate и обучение с PackedSequence](#section-14)
15. [Одномерные свёртки](#section-15)
16. [Общая модель последовательности](#section-16)
17. [От последовательностей к CRNN](#section-17)
18. [CTC и декодирование](#section-18)
19. [Обучение OCR и проверка распознаваний](#section-19)
20. [Итоги и следующий шаг](#section-20)

[Приложение. STN](#stn)

In [ ]:
from pathlib import Path
from runpy import run_path
import os

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config.py').is_file()), None)
if root is None:
    try:
        import google.colab
    except ModuleNotFoundError:
        raise RuntimeError('Откройте notebook из репозитория курса.') from None
    import requests
    url = 'https://raw.githubusercontent.com/SergeyMalashenko/2026_ML3/main/config.py'
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    Path('config.py').write_bytes(response.content)
    root = Path.cwd()
course = run_path(str(root / 'config.py'))
seminar_plots = course['config_seminar08']()

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

device = course['select_device'](os.environ.get('AI_MASTERS_DEVICE', 'auto'))
torch.manual_seed(42)
seminar_plots.configure_plots()
print('Устройство:', device)

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch.nn.utils import rnn as rnn_utils
generate_sequences = seminar_plots.generate_sequences

<a id="section-1"></a>

## 1. Последовательности и данные квадрата

**Вопрос перед кодом:** На рисунке квадрат без стрелок. Можно ли определить, в какую сторону его рисовали?

### Генерация данных

In [ ]:
fig = seminar_plots.godoy_counter_vs_clock(draw_arrows=False);

In [ ]:
fig = seminar_plots.godoy_counter_vs_clock();

In [ ]:
fig = seminar_plots.godoy_plot_sequences();

In [ ]:
points, directions = generate_sequences(n=128, seed=13)

In [ ]:
fig = seminar_plots.godoy_plot_data(points, directions);

<a id="section-2"></a>

## 2. RNN: от состояния к ячейке и слою

**Вопрос перед кодом:** Что хранится в состоянии, а что является обучаемыми параметрами?

In [ ]:
hidden_state = torch.zeros(2)
hidden_state

In [ ]:
seminar_plots.plot_chapter8_diagram('rnn2');

### Рекуррентная ячейка

In [ ]:
seminar_plots.plot_chapter8_diagram('rnn1');

$$
\begin{aligned}
\color{#0066CC}{t_h} &= \color{#0066CC}{W_{hh}h_{t-1}+b_{hh}} \\
\color{red}{t_x} &= \color{red}{W_{ih}x_t+b_{ih}} \\
\color{#0066CC}{h_t} &= \color{#0066CC}{\tanh(t_h+t_x)}
\end{aligned}
$$

In [ ]:
seminar_plots.plot_chapter8_diagram('rnn_cell_diagram');

In [ ]:
n_features = 2
hidden_dim = 2

torch.manual_seed(19)
rnn_cell = nn.RNNCell(input_size=n_features, hidden_size=hidden_dim)
rnn_state = rnn_cell.state_dict()
rnn_state

In [ ]:
linear_input = nn.Linear(n_features, hidden_dim)
linear_hidden = nn.Linear(hidden_dim, hidden_dim)

with torch.no_grad():
    linear_input.weight = nn.Parameter(rnn_state['weight_ih'])
    linear_input.bias = nn.Parameter(rnn_state['bias_ih'])
    linear_hidden.weight = nn.Parameter(rnn_state['weight_hh'])
    linear_hidden.bias = nn.Parameter(rnn_state['bias_hh'])

In [ ]:
initial_hidden = torch.zeros(1, hidden_dim)
initial_hidden

In [ ]:
th = linear_hidden(initial_hidden)
th

In [ ]:
X = torch.as_tensor(points[0]).float()
X

In [ ]:
tx = linear_input(X[0:1])
tx

In [ ]:
adding = th + tx
adding

In [ ]:
torch.tanh(adding)

In [ ]:
rnn_cell(X[0:1])

In [ ]:
torch.testing.assert_close(torch.tanh(adding), rnn_cell(X[0:1]))

In [ ]:
fig = seminar_plots.godoy_figure8(linear_hidden, linear_input, X);

In [ ]:
# Это батч отдельных точек, а не четыре последовательных шага
rnn_cell(X)

In [ ]:
hidden = torch.zeros(1, hidden_dim)
for i in range(X.shape[0]):
    out = rnn_cell(X[i:i+1], hidden)
    print(out)
    hidden = out

### T2. Следующее состояние

Вычислите состояние после второй точки вручную через `F.linear` и `tanh`. Не создавайте новые веса.

In [ ]:
task_x = X[1:2]
task_h = rnn_cell(X[:1]).detach()
# Ваш код здесь


<details>
<summary>Решение и проверка</summary>

```python
task_next = torch.tanh(linear_input(task_x) + linear_hidden(task_h))
torch.testing.assert_close(task_next, rnn_cell(task_x, task_h))
```

</details>

In [ ]:
seminar_plots.plot_chapter8_diagram('rnn_cell_diagram_seq');

### Рекуррентный слой

In [ ]:
n_features = 2
hidden_dim = 2

torch.manual_seed(19)
rnn = nn.RNN(input_size=n_features, hidden_size=hidden_dim)
rnn.state_dict()

### Формы тензоров

In [ ]:
batch = torch.as_tensor(np.array(points[:3])).float()
batch.shape

In [ ]:
permuted_batch = batch.permute(1, 0, 2)
permuted_batch.shape

In [ ]:
torch.manual_seed(19)
rnn = nn.RNN(input_size=n_features, hidden_size=hidden_dim)
out, final_hidden = rnn(permuted_batch)
out.shape, final_hidden.shape

In [ ]:
(out[-1] == final_hidden).all()

In [ ]:
batch_hidden = final_hidden.permute(1, 0, 2)
batch_hidden.shape

In [ ]:
torch.manual_seed(19)
rnn_batch_first = nn.RNN(input_size=n_features, hidden_size=hidden_dim, batch_first=True)
out, final_hidden = rnn_batch_first(batch)
out.shape, final_hidden.shape

<a id="section-3"></a>

## 3. Многослойная RNN

**Вопрос перед кодом:** Второй слой получает исходные точки или выходы первого слоя?

In [ ]:
seminar_plots.plot_chapter8_diagram('stacked_rnn');

In [ ]:
torch.manual_seed(19)
rnn_stacked = nn.RNN(input_size=2, hidden_size=2, num_layers=2, batch_first=True)
state = rnn_stacked.state_dict()
state

In [ ]:
rnn_layer0 = nn.RNN(input_size=2, hidden_size=2, batch_first=True)
rnn_layer1 = nn.RNN(input_size=2, hidden_size=2, batch_first=True)

rnn_layer0.load_state_dict(dict(list(state.items())[:4]))
rnn_layer1.load_state_dict(dict([(k[:-1]+'0', v) for k, v in list(state.items())[4:]]))

In [ ]:
x = torch.as_tensor(np.array(points[0:1])).float()

In [ ]:
out0, h0 = rnn_layer0(x)

In [ ]:
out1, h1 = rnn_layer1(out0)

In [ ]:
out1, torch.cat([h0, h1])

In [ ]:
out, hidden = rnn_stacked(x)
out, hidden

In [ ]:
(out[:, -1] == hidden.permute(1, 0, 2)[:, -1]).all()

In [ ]:
torch.testing.assert_close(out, out1)
torch.testing.assert_close(hidden, torch.cat([h0, h1]))

<a id="section-4"></a>

## 4. Двунаправленная RNN

**Вопрос перед кодом:** Почему последний output не содержит оба конечных состояния?

In [ ]:
seminar_plots.plot_chapter8_diagram('bidirect_rnn');

In [ ]:
torch.manual_seed(19)
rnn_bidirect = nn.RNN(input_size=2, hidden_size=2, bidirectional=True, batch_first=True)
state = rnn_bidirect.state_dict()
state

In [ ]:
rnn_forward = nn.RNN(input_size=2, hidden_size=2, batch_first=True)
rnn_reverse = nn.RNN(input_size=2, hidden_size=2, batch_first=True)

rnn_forward.load_state_dict(dict(list(state.items())[:4]))
rnn_reverse.load_state_dict(dict([(k[:-8], v) for k, v in list(state.items())[4:]]))

In [ ]:
x_rev = torch.flip(x, dims=[1]) # N, L, F
x_rev

In [ ]:
out, h = rnn_forward(x)
out_rev, h_rev = rnn_reverse(x_rev)
out_rev_back = torch.flip(out_rev, dims=[1])

In [ ]:
torch.cat([out, out_rev_back], dim=2), torch.cat([h, h_rev])

In [ ]:
out, hidden = rnn_bidirect(x)
out, hidden

In [ ]:
out[:, -1] == hidden.permute(1, 0, 2).view(1, -1)

In [ ]:
torch.testing.assert_close(out[:, :, :2], rnn_forward(x)[0])
torch.testing.assert_close(out[:, :, 2:], out_rev_back)
torch.testing.assert_close(hidden[0], out[:, -1, :2])
torch.testing.assert_close(hidden[1], out[:, 0, 2:])

<a id="section-5"></a>

## 5. Square Model: обучение RNN

**Вопрос перед кодом:** Почему для классификации одной последовательности достаточно одного логита?

### Генерация данных

In [ ]:
test_points, test_directions = generate_sequences(seed=19)

### Подготовка данных

In [ ]:
train_data = TensorDataset(torch.as_tensor(np.array(points)).float(), 
                           torch.as_tensor(directions).view(-1, 1).float())
test_data = TensorDataset(torch.as_tensor(np.array(test_points)).float(),
                          torch.as_tensor(test_directions).view(-1, 1).float())

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

### Конфигурация модели

In [ ]:
class SquareModel(nn.Module):
    def __init__(self, n_features, hidden_dim, n_outputs):
        super(SquareModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_features = n_features
        self.n_outputs = n_outputs
        self.hidden = None
        # Simple RNN
        self.basic_rnn = nn.RNN(self.n_features, self.hidden_dim, batch_first=True)
        # Classifier to produce as many logits as outputs
        self.classifier = nn.Linear(self.hidden_dim, self.n_outputs)
                
    def forward(self, X):
        # X is batch first (N, L, F)
        # output is (N, L, H)
        # final hidden state is (1, N, H)
        batch_first_output, self.hidden = self.basic_rnn(X)
        
        # Последний выход: (N, H)
        last_output = batch_first_output[:, -1]
        # Логиты: (N, n_outputs)
        out = self.classifier(last_output)
        
        # final output is (N, n_outputs)
        return out.view(-1, self.n_outputs)

In [ ]:
torch.manual_seed(21)
model = SquareModel(n_features=2, hidden_dim=2, n_outputs=1)
loss = nn.BCEWithLogitsLoss()
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

### Обучение модели

Годой использует StepByStep. Здесь тот же цикл выписан обычным PyTorch и повторно применяется к следующим моделям. Число эпох, seed и learning rate сохранены. Авторский `test_loader` используется для наблюдения за обучением, то есть как validation; независимую финальную тестовую оценку здесь не заявляем. После обучения модель явно возвращается на CPU для маленьких диагностических графиков.

In [ ]:
def evaluate_sequence_model(model, loader, loss_fn, training_device):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(training_device), y_batch.to(training_device)
            logits = model(X_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += ((logits >= 0) == (y_batch >= 0.5)).sum().item()
            count += len(y_batch)
    return {'loss': total_loss / count, 'accuracy': correct / count, 'count': count}

def fit_sequence_model(model, loss_fn, optimizer, train_loader, test_loader=None, n_epochs=100):
    # Обычный цикл вместо StepByStep. Диагностика после обучения явно на CPU.
    torch.manual_seed(42)
    model.to(device)
    assert {id(p) for g in optimizer.param_groups for p in g['params']} == {id(p) for p in model.parameters()}
    history = {'train': [], 'validation': []}
    for epoch in range(n_epochs):
        model.train()
        total_loss, count = 0.0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(X_batch)
            batch_loss = loss_fn(logits, y_batch)
            batch_loss.backward()
            optimizer.step()
            total_loss += batch_loss.item() * len(y_batch)
            count += len(y_batch)
        history['train'].append(total_loss / count)
        if test_loader is not None:
            history['validation'].append(evaluate_sequence_model(model, test_loader, loss_fn, device)['loss'])
    model.cpu().eval()
    return history

In [ ]:
history_rnn = fit_sequence_model(model, loss, optimizer, train_loader, test_loader, n_epochs=100)
model_rnn = model

In [ ]:
seminar_plots.plot_sequence_losses(history_rnn, 'RNN');

In [ ]:
metrics_rnn = evaluate_sequence_model(model_rnn, test_loader, loss, torch.device('cpu'))
metrics_rnn

<a id="section-6"></a>

## 6. Что происходит со входами и состояниями

**Вопрос перед кодом:** Что на этих графиках обучается, а что меняется от точки к точке?

### Преобразованные входы

In [ ]:
state = model.basic_rnn.state_dict()
state['weight_ih_l0'], state['bias_ih_l0']

In [ ]:
fig = seminar_plots.godoy_figure13(model.basic_rnn);

### Скрытые состояния

In [ ]:
fig = seminar_plots.godoy_canonical_contour(model);

In [ ]:
fig = seminar_plots.godoy_hidden_states_contour(model, points, directions);

### Путь скрытого состояния

In [ ]:
fig = seminar_plots.godoy_figure16(model.basic_rnn);

In [ ]:
square = torch.tensor([[-1, -1], [-1, 1], [1, 1], [1, -1]]).float().view(1, 4, 2)
model(square)
model.hidden

In [ ]:
fig = seminar_plots.godoy_figure17(model.basic_rnn);

<a id="section-7"></a>

## 7. GRU: гейты и ручной шаг

**Вопрос перед кодом:** Как получить почти неизменное состояние с помощью update gate?

$$
\begin{aligned}
h_{\text{new}} &= \tanh(t_h+t_x) \\
h' &= h_{\text{new}}*(1-z)+h_{\text{old}}*z
\end{aligned}
$$

$$
\Large
h_{\text{new}} = \text{tanh}(r*t_h + t_x)
$$

$$
\Large
h' = \text{tanh}(r*t_h + t_x) * (1-z) + h*z
$$

### Ячейка GRU

$$
\begin{aligned}
\text{RNN:}\quad h' &= \tanh(t_h+t_x) \\
\text{GRU:}\quad h' &= \underbrace{\underbrace{\tanh(\color{red}{r}*t_{hn}+t_{xn})}_{n}*\color{#0066CC}{(1-z)}+\color{gray}{h}*\color{#0066CC}{z}}_{\text{взвешенная сумма }n\text{ и }h}
\end{aligned}
$$

$$
\begin{aligned}
\color{red}{r\ (\text{сброс})} &= \color{red}{\sigma(t_{hr}+t_{xr})} \\
\color{#0066CC}{z\ (\text{обновление})} &= \color{#0066CC}{\sigma(t_{hz}+t_{xz})} \\
n &= \tanh(r*t_{hn}+t_{xn})
\end{aligned}
$$

$$
\begin{aligned}
\color{red}{t_{hr}\ (\text{состояние})} &= \color{red}{W_{hr}h+b_{hr}} \\
\color{red}{t_{xr}\ (\text{вход})} &= \color{red}{W_{ir}x+b_{ir}} \\
\color{#0066CC}{t_{hz}\ (\text{состояние})} &= \color{#0066CC}{W_{hz}h+b_{hz}} \\
\color{#0066CC}{t_{xz}\ (\text{вход})} &= \color{#0066CC}{W_{iz}x+b_{iz}} \\
\color{black}{t_{hn}\ (\text{состояние})} &= \color{black}{W_{hn}h+b_{hn}} \\
\color{black}{t_{xn}\ (\text{вход})} &= \color{black}{W_{in}x+b_{in}}
\end{aligned}
$$

In [ ]:
seminar_plots.plot_chapter8_diagram('gru_cell');

In [ ]:
n_features = 2
hidden_dim = 2

torch.manual_seed(17)
gru_cell = nn.GRUCell(input_size=n_features, hidden_size=hidden_dim)
gru_state = gru_cell.state_dict()
gru_state

In [ ]:
Wx, bx = gru_state['weight_ih'], gru_state['bias_ih']
Wh, bh = gru_state['weight_hh'], gru_state['bias_hh']

print(Wx.shape, Wh.shape)
print(bx.shape, bh.shape)

$$
\begin{aligned}
W_{xr} &= \begin{bmatrix}-0.0930 & 0.0497 \\ 0.4670 & -0.5319\end{bmatrix} \\[6pt]
W_{xz} &= \begin{bmatrix}-0.6656 & 0.0699 \\ -0.1662 & 0.0654\end{bmatrix} \\[6pt]
W_{xn} &= \begin{bmatrix}-0.0449 & -0.6828 \\ -0.6769 & -0.1889\end{bmatrix}
\end{aligned}
$$

$$
\Large
\underbrace{-0.4316,  0.4019}_{b_{xr}}, 
\underbrace{0.1222, -0.4647}_{b_{xz}}, 
\underbrace{-0.5578,  0.4493}_{b_{xn}}
$$

In [ ]:
Wxr, Wxz, Wxn = Wx.split(hidden_dim, dim=0)
bxr, bxz, bxn = bx.split(hidden_dim, dim=0)

Whr, Whz, Whn = Wh.split(hidden_dim, dim=0)
bhr, bhz, bhn = bh.split(hidden_dim, dim=0)

Wxr, bxr

In [ ]:
def linear_layers(Wx, bx, Wh, bh):
    hidden_dim, n_features = Wx.size()    
    lin_input = nn.Linear(n_features, hidden_dim)
    lin_input.load_state_dict({'weight': Wx, 'bias': bx})
    lin_hidden = nn.Linear(hidden_dim, hidden_dim)
    lin_hidden.load_state_dict({'weight': Wh, 'bias': bh})
    return lin_hidden, lin_input

r_hidden, r_input = linear_layers(Wxr, bxr, Whr, bhr) # reset gate - red
z_hidden, z_input = linear_layers(Wxz, bxz, Whz, bhz) # update gate - blue
n_hidden, n_input = linear_layers(Wxn, bxn, Whn, bhn) # candidate state - black

In [ ]:
def reset_gate(h, x):
    thr = r_hidden(h)
    txr = r_input(x)
    r = torch.sigmoid(thr + txr)
    return r  # red
    
def update_gate(h, x):
    thz = z_hidden(h)
    txz = z_input(x)
    z = torch.sigmoid(thz + txz)
    return z  # blue
    
def candidate_n(h, x, r):
    thn = n_hidden(h)
    txn = n_input(x)
    n = torch.tanh(r * thn + txn)
    return n  # black

In [ ]:
initial_hidden = torch.zeros(1, hidden_dim)
X = torch.as_tensor(points[0]).float()
first_corner = X[0:1]

In [ ]:
r = reset_gate(initial_hidden, first_corner)
r

In [ ]:
n = candidate_n(initial_hidden, first_corner, r)
n

In [ ]:
z = update_gate(initial_hidden, first_corner)
z

In [ ]:
h_prime = n*(1-z) + initial_hidden*z
h_prime

In [ ]:
gru_cell(first_corner)

In [ ]:
torch.testing.assert_close(h_prime, gru_cell(first_corner))

### Слой GRU

<a id="section-8"></a>

## 8. Square Model II: обучение GRU

**Вопрос перед кодом:** Какая часть цикла обучения изменилась после замены RNN на GRU?

In [ ]:
class SquareModelGRU(nn.Module):
    def __init__(self, n_features, hidden_dim, n_outputs):
        super(SquareModelGRU, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_features = n_features
        self.n_outputs = n_outputs
        self.hidden = None
        # Simple GRU
        self.basic_rnn = nn.GRU(self.n_features, self.hidden_dim, batch_first=True)
        # Classifier to produce as many logits as outputs
        self.classifier = nn.Linear(self.hidden_dim, self.n_outputs)
                
    def forward(self, X):
        # X is batch first (N, L, F)
        # output is (N, L, H)
        # final hidden state is (1, N, H)
        batch_first_output, self.hidden = self.basic_rnn(X)
        
        # Последний выход: (N, H)
        last_output = batch_first_output[:, -1]
        # Логиты: (N, n_outputs)
        out = self.classifier(last_output)
        
        # final output is (N, n_outputs)
        return out.view(-1, self.n_outputs)

### Конфигурация и обучение

In [ ]:
torch.manual_seed(21)
model = SquareModelGRU(n_features=2, hidden_dim=2, n_outputs=1)
loss = nn.BCEWithLogitsLoss()
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
history_gru = fit_sequence_model(model, loss, optimizer, train_loader, test_loader, n_epochs=100)
model_gru = model

In [ ]:
seminar_plots.plot_sequence_losses(history_gru, 'GRU');

In [ ]:
metrics_gru = evaluate_sequence_model(model_gru, test_loader, loss, torch.device('cpu'))
metrics_gru

<a id="section-9"></a>

## 9. Визуализация GRU

**Вопрос перед кодом:** Как гейт меняет путь состояния, не меняя размер самого состояния?

### Скрытые состояния

In [ ]:
fig = seminar_plots.godoy_figure20(model_rnn, model_gru);

In [ ]:
fig = seminar_plots.godoy_hidden_states_contour(model, points, directions);

### Путь состояния с гейтами

In [ ]:
fig = seminar_plots.godoy_figure22(model.basic_rnn);

<a id="section-10"></a>

## 10. LSTM: два состояния и ручной шаг

**Вопрос перед кодом:** Зачем LSTM передавать дальше два состояния, а не одно?

$$
\Large
g = \text{tanh}(t_{hg}+t_{xg})
$$

$$
\Large
c' = g * i + c * f
$$

$$
\Large
h' = \text{tanh}(c') * o
$$

### Ячейка LSTM

$$
\begin{aligned}
\text{RNN:}\quad h' &= \tanh(t_h+t_x) \\
\text{GRU:}\quad h' &= \tanh(r*t_{hn}+t_{xn})*(1-z)+h*z \\
\text{LSTM:}\quad c' &= \underbrace{\tanh(t_{hg}+t_{xg})}_{g}*\color{#82b366}{i}+\color{gray}{c}*\color{red}{f} \\
h' &= \tanh(c')*\color{#0066CC}{o}
\end{aligned}
$$

$$
\begin{aligned}
\color{#82b366}{i\ (\text{вход})} &= \color{#82b366}{\sigma(t_{hi}+t_{xi})} \\
\color{red}{f\ (\text{забывание})} &= \color{red}{\sigma(t_{hf}+t_{xf})} \\
\color{#0066CC}{o\ (\text{выход})} &= \color{#0066CC}{\sigma(t_{ho}+t_{xo})} \\
g &= \tanh(t_{hg}+t_{xg})
\end{aligned}
$$

$$
\begin{aligned}
\color{#82b366}{t_{hi}\ (\text{состояние})} &= \color{#82b366}{W_{hi}h+b_{hi}} \\
\color{#82b366}{t_{xi}\ (\text{вход})} &= \color{#82b366}{W_{ii}x+b_{ii}} \\
\color{red}{t_{hf}\ (\text{состояние})} &= \color{red}{W_{hf}h+b_{hf}} \\
\color{red}{t_{xf}\ (\text{вход})} &= \color{red}{W_{if}x+b_{if}} \\
\color{black}{t_{hg}\ (\text{состояние})} &= \color{black}{W_{hg}h+b_{hg}} \\
\color{black}{t_{xg}\ (\text{вход})} &= \color{black}{W_{ig}x+b_{ig}} \\
\color{#0066CC}{t_{ho}\ (\text{состояние})} &= \color{#0066CC}{W_{ho}h+b_{ho}} \\
\color{#0066CC}{t_{xo}\ (\text{вход})} &= \color{#0066CC}{W_{io}x+b_{io}}
\end{aligned}
$$

In [ ]:
seminar_plots.plot_chapter8_diagram('lstm_cell');

In [ ]:
n_features = 2
hidden_dim = 2

torch.manual_seed(17)
lstm_cell = nn.LSTMCell(input_size=n_features, hidden_size=hidden_dim)
lstm_state = lstm_cell.state_dict()
lstm_state

In [ ]:
Wx, bx = lstm_state['weight_ih'], lstm_state['bias_ih']
Wh, bh = lstm_state['weight_hh'], lstm_state['bias_hh']

# Split weights and biases for data points
Wxi, Wxf, Wxg, Wxo = Wx.split(hidden_dim, dim=0)
bxi, bxf, bxg, bxo = bx.split(hidden_dim, dim=0)
# Split weights and biases for hidden state
Whi, Whf, Whg, Who = Wh.split(hidden_dim, dim=0)
bhi, bhf, bhg, bho = bh.split(hidden_dim, dim=0)

# Creates linear layers for the components
i_hidden, i_input = linear_layers(Wxi, bxi, Whi, bhi) # input gate - green
f_hidden, f_input = linear_layers(Wxf, bxf, Whf, bhf) # forget gate - red
o_hidden, o_input = linear_layers(Wxo, bxo, Who, bho) # output gate - blue

In [ ]:
g_cell = nn.RNNCell(n_features, hidden_dim) # black
g_cell.load_state_dict({'weight_ih': Wxg, 'bias_ih': bxg,
                        'weight_hh': Whg, 'bias_hh': bhg})

In [ ]:
def forget_gate(h, x):
    thf = f_hidden(h)
    txf = f_input(x)
    f = torch.sigmoid(thf + txf)
    return f  # red
    
def output_gate(h, x):
    tho = o_hidden(h)
    txo = o_input(x)
    o = torch.sigmoid(tho + txo)
    return o  # blue

def input_gate(h, x):
    thi = i_hidden(h)
    txi = i_input(x)
    i = torch.sigmoid(thi + txi)
    return i  # green

In [ ]:
initial_hidden = torch.zeros(1, hidden_dim)
initial_cell = torch.zeros(1, hidden_dim)

X = torch.as_tensor(points[0]).float()
first_corner = X[0:1]

In [ ]:
g = g_cell(first_corner)
i = input_gate(initial_hidden, first_corner)
gated_input = g * i
gated_input

In [ ]:
f = forget_gate(initial_hidden, first_corner)
gated_cell = initial_cell * f
gated_cell

In [ ]:
c_prime = gated_cell + gated_input
c_prime

In [ ]:
o = output_gate(initial_hidden, first_corner)
h_prime = o * torch.tanh(c_prime)
h_prime

In [ ]:
(h_prime, c_prime)

In [ ]:
lstm_cell(first_corner)

In [ ]:
actual_h, actual_c = lstm_cell(first_corner)
torch.testing.assert_close(h_prime, actual_h)
torch.testing.assert_close(c_prime, actual_c)

### Слой LSTM

<a id="section-11"></a>

## 11. Square Model III: обучение LSTM

**Вопрос перед кодом:** Что нужно изменить в forward при переходе от GRU к LSTM?

### Конфигурация модели

In [ ]:
class SquareModelLSTM(nn.Module):
    def __init__(self, n_features, hidden_dim, n_outputs):
        super(SquareModelLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_features = n_features
        self.n_outputs = n_outputs
        self.hidden = None
        self.cell = None
        # Simple LSTM
        self.basic_rnn = nn.LSTM(self.n_features, self.hidden_dim, batch_first=True) 
        # Classifier to produce as many logits as outputs        
        self.classifier = nn.Linear(self.hidden_dim, self.n_outputs)
                
    def forward(self, X):
        # X is batch first (N, L, F)
        # output is (N, L, H)
        # final hidden state is (1, N, H)
        # final cell state is (1, N, H)
        batch_first_output, (self.hidden, self.cell) = self.basic_rnn(X)
        
        # Последний выход: (N, H)
        last_output = batch_first_output[:, -1]
        # Логиты: (N, n_outputs)
        out = self.classifier(last_output)
        
        # final output is (N, n_outputs)
        return out.view(-1, self.n_outputs)

### Конфигурация и обучение

In [ ]:
torch.manual_seed(21)
model = SquareModelLSTM(n_features=2, hidden_dim=2, n_outputs=1)
loss = nn.BCEWithLogitsLoss()
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
history_lstm = fit_sequence_model(model, loss, optimizer, train_loader, test_loader, n_epochs=100)
model_lstm = model

In [ ]:
seminar_plots.plot_sequence_losses(history_lstm, 'LSTM');

In [ ]:
metrics_lstm = evaluate_sequence_model(model_lstm, test_loader, loss, torch.device('cpu'))
metrics_lstm

<a id="section-12"></a>

## 12. Сравнение состояний RNN, GRU и LSTM

**Вопрос перед кодом:** Доказывает ли один запуск, что один тип рекуррентной сети всегда лучше?

In [ ]:
fig = seminar_plots.godoy_figure25(model_rnn, model_gru, model_lstm);

In [ ]:
fig = seminar_plots.godoy_hidden_states_contour(model, points, directions);

<a id="section-13"></a>

## 13. Последовательности переменной длины

**Вопрос перед кодом:** Почему нулевое дополнение не означает, что RNN перестанет обновлять состояние?

In [ ]:
s0 = points[0]      # 4 data points
s1 = points[1][2:]  # 2 data points
s2 = points[2][1:]  # 3 data points

s0.shape, s1.shape, s2.shape

### Дополнение последовательностей

In [ ]:
all_seqs = [s0, s1, s2]
#torch.as_tensor(all_seqs)

In [ ]:
seq_tensors = [torch.as_tensor(seq).float() for seq in all_seqs]
padded = rnn_utils.pad_sequence(seq_tensors, batch_first=True)
padded

In [ ]:
torch.manual_seed(11)
rnn = nn.RNN(2, 2, batch_first=True)

In [ ]:
output_padded, hidden_padded = rnn(padded)
output_padded

In [ ]:
hidden_padded.permute(1, 0, 2)

### Упаковка последовательностей

In [ ]:
packed = rnn_utils.pack_sequence(seq_tensors, enforce_sorted=False)
packed

In [ ]:
seminar_plots.plot_chapter8_diagram('packed_seq_inddata');

In [ ]:
seminar_plots.plot_chapter8_diagram('packed_seq_data');

In [ ]:
(packed.data[[0, 3, 6, 8]] == seq_tensors[0]).all()

In [ ]:
output_packed, hidden_packed = rnn(packed)
output_packed, hidden_packed

In [ ]:
hidden_packed == hidden_padded

In [ ]:
output_packed.data[[2, 5]] # x1 sequence

### Распаковка с дополнением

In [ ]:
output_unpacked, seq_sizes = rnn_utils.pad_packed_sequence(output_packed, batch_first=True)
output_unpacked, seq_sizes

In [ ]:
output_unpacked[:, -1]

In [ ]:
seq_idx = torch.arange(seq_sizes.size(0))
output_unpacked[seq_idx, seq_sizes-1]

### Упаковка дополненного батча

In [ ]:
len_seqs = [len(seq) for seq in all_seqs]
len_seqs

In [ ]:
packed = rnn_utils.pack_padded_sequence(padded, len_seqs, enforce_sorted=False, batch_first=True)
packed

<a id="section-14"></a>

## 14. Dataset, collate и обучение с PackedSequence

**Вопрос перед кодом:** Что должен вернуть collate, если длины последовательностей различаются?

In [ ]:
var_points, var_directions = generate_sequences(variable_len=True)
var_points[:2]

### Подготовка данных

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.x = [torch.as_tensor(s).float() for s in x]
        self.y = torch.as_tensor(y).float().view(-1, 1)
        
    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)
    
train_var_data = CustomDataset(var_points, var_directions)

In [ ]:
train_var_loader = DataLoader(train_var_data, batch_size=16, shuffle=True)
#next(iter(train_var_loader))

### Сборка батча

In [ ]:
def pack_collate(batch):
    X = [item[0] for item in batch]
    y = [item[1] for item in batch]
    X_pack = rnn_utils.pack_sequence(X, enforce_sorted=False)

    return X_pack, torch.as_tensor(y).view(-1, 1)

In [ ]:
# list of tuples returned by the dataset
dummy_batch = [train_var_data[0], train_var_data[1]]
dummy_x, dummy_y = pack_collate(dummy_batch)
dummy_x

In [ ]:
train_var_loader = DataLoader(train_var_data, batch_size=16, shuffle=True, collate_fn=pack_collate)
x_batch, y_batch = next(iter(train_var_loader))

### Square Model IV: упакованные последовательности

In [ ]:
class SquareModelPacked(nn.Module):
    def __init__(self, n_features, hidden_dim, n_outputs):
        super(SquareModelPacked, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_features = n_features
        self.n_outputs = n_outputs
        self.hidden = None
        self.cell = None
        # Simple LSTM
        self.basic_rnn = nn.LSTM(self.n_features, self.hidden_dim, bidirectional=True) 
        # Classifier to produce as many logits as outputs        
        self.classifier = nn.Linear(2 * self.hidden_dim, self.n_outputs)

    def forward(self, X):       
        # X is a PACKED sequence now
        
        # output is PACKED
        # final hidden state is (2, N, H) - bidirectional
        # final cell state is (2, N, H) - bidirectional
        rnn_out, (self.hidden, self.cell) = self.basic_rnn(X)
        # unpack the output (N, L, 2*H)
        batch_first_output, seq_sizes = rnn_utils.pad_packed_sequence(rnn_out, batch_first=True)
        
        # Последний выход: (N, 2*H)
        seq_idx = torch.arange(seq_sizes.size(0))
        last_output = batch_first_output[seq_idx, seq_sizes-1]
        # Логиты: (N, n_outputs)
        out = self.classifier(last_output)
                
        # final output is (N, n_outputs)
        return out.view(-1, self.n_outputs)

Авторская модель выбирает последний действительный **output**, включая его обратную часть. Это допустимый readout, но не конкатенация двух конечных состояний: обратное конечное состояние находится на первом шаге. Сохраняем пример Годоя и не называем этот вектор двумя конечными состояниями.

### Конфигурация и обучение

In [ ]:
torch.manual_seed(21)
model = SquareModelPacked(n_features=2, hidden_dim=2, n_outputs=1)
loss = nn.BCEWithLogitsLoss()
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
history_packed = fit_sequence_model(model, loss, optimizer, train_var_loader, None, n_epochs=100)
model_packed = model

In [ ]:
seminar_plots.plot_sequence_losses(history_packed, 'PACKED');

In [ ]:
metrics_packed = evaluate_sequence_model(model_packed, train_var_loader, loss, torch.device('cpu'))
metrics_packed

<a id="section-15"></a>

## 15. Одномерные свёртки

**Вопрос перед кодом:** Почему для Conv1d меняем местами оси длины и признаков?

In [ ]:
temperatures = np.array([5, 11, 15, 6, 5, 3, 3, 0, 0, 3, 4, 2, 1])

In [ ]:
seminar_plots.plot_chapter8_diagram('conv1_ma');

In [ ]:
size = 5
weight = torch.ones(size) * 0.2
F.conv1d(torch.as_tensor(temperatures).float().view(1, 1, -1), 
         weight=weight.view(1, 1, -1))

### Формы тензоров

In [ ]:
seqs = torch.as_tensor(np.array(points)).float() # N, L, F
seqs_length_last = seqs.permute(0, 2, 1) 
seqs_length_last.shape # N, F=C, L

### Несколько признаков, или каналов

In [ ]:
torch.manual_seed(17)
conv_seq = nn.Conv1d(in_channels=2, out_channels=1, kernel_size=2, bias=False)
conv_seq.weight, conv_seq.weight.shape

In [ ]:
seminar_plots.plot_chapter8_diagram('conv1d');

In [ ]:
conv_seq(seqs_length_last[0:1])

$$L_{out}=\left\lfloor\frac{L_{in}+2p-d(k-1)-1}{s}+1\right\rfloor.$$

В текущем примере dilation d=1; k — размер ядра.

### Разреженное ядро

In [ ]:
seminar_plots.plot_chapter8_diagram('conv1d_dilated');

In [ ]:
torch.manual_seed(17)
conv_dilated = nn.Conv1d(in_channels=2, out_channels=1, kernel_size=2, dilation=2, bias=False)
conv_dilated.weight, conv_dilated.weight.shape

In [ ]:
conv_dilated(seqs_length_last[0:1])

$$L_{out}=\left\lfloor\frac{L_{in}+2p-d(k-1)-1}{s}+1\right\rfloor.$$

### Подготовка данных

In [ ]:
train_data = TensorDataset(torch.as_tensor(np.array(points)).float().permute(0, 2, 1), 
                           torch.as_tensor(directions).view(-1, 1).float())
test_data = TensorDataset(torch.as_tensor(np.array(test_points)).float().permute(0, 2, 1), 
                          torch.as_tensor(test_directions).view(-1, 1).float())

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
test_loader = DataLoader(test_data, batch_size=16)

### Конфигурация и обучение

In [ ]:
torch.manual_seed(21)
model = nn.Sequential()
model.add_module('conv1d', nn.Conv1d(2, 1, kernel_size=2))
model.add_module('relu', nn.ReLU())
model.add_module('flatten', nn.Flatten())
model.add_module('output', nn.Linear(3, 1))
loss = nn.BCEWithLogitsLoss()
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
history_conv1 = fit_sequence_model(model, loss, optimizer, train_loader, test_loader, n_epochs=100)
model_conv1 = model

In [ ]:
seminar_plots.plot_sequence_losses(history_conv1, 'CONV1');

In [ ]:
metrics_conv1 = evaluate_sequence_model(model_conv1, test_loader, loss, torch.device('cpu'))
metrics_conv1

### Визуализация модели

In [ ]:
model.conv1d.state_dict()

In [ ]:
seminar_plots.plot_chapter8_diagram('conv1d_edges');

$$a=b+\sum_{c=0}^{1}\sum_{j=0}^{1}w_{c,j}x_{c,t+j}.$$

Смотрим на фактически обученные веса выше. Численные коэффициенты из книги не подставляем вместо результата текущего запуска.

**Вопрос:** какой переход между соседними вершинами даёт большую активацию при полученных весах? Один фильтр на одном запуске не доказывает преимущество Conv1d над рекуррентной сетью.

<a id="section-16"></a>

## 16. Общая модель последовательности

**Вопрос перед кодом:** Как различить последний выход и конечные состояния двух направлений?

### Выборка фиксированной длины

In [ ]:
points, directions = generate_sequences(n=128, seed=13)
train_data = TensorDataset(torch.as_tensor(np.array(points)).float(), 
                           torch.as_tensor(directions).view(-1, 1).float())
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)

### Выборка переменной длины

In [ ]:
var_points, var_directions = generate_sequences(variable_len=True)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, x, y):
        self.x = [torch.as_tensor(s).float() for s in x]
        self.y = torch.as_tensor(y).float().view(-1, 1)
        
    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)
    
train_var_data = CustomDataset(var_points, var_directions)

In [ ]:
def pack_collate(batch):
    X = [item[0] for item in batch]
    y = [item[1] for item in batch]
    X_pack = rnn_utils.pack_sequence(X, enforce_sorted=False)

    return X_pack, torch.as_tensor(y).view(-1, 1)

train_var_loader = DataLoader(train_var_data, batch_size=16, shuffle=True, collate_fn=pack_collate)

### Одна модель для разных рекуррентных слоёв

In [ ]:
class SquareModelOne(nn.Module):
    def __init__(self, n_features, hidden_dim, n_outputs, rnn_layer=nn.LSTM, **kwargs):
        super(SquareModelOne, self).__init__()
        self.hidden_dim = hidden_dim
        self.n_features = n_features
        self.n_outputs = n_outputs
        self.hidden = None
        self.cell = None
        self.basic_rnn = rnn_layer(self.n_features, self.hidden_dim, batch_first=True, **kwargs)
        output_dim = (self.basic_rnn.bidirectional + 1) * self.hidden_dim
        # Classifier to produce as many logits as outputs
        self.classifier = nn.Linear(output_dim, self.n_outputs)
        
    def forward(self, X):
        is_packed = isinstance(X, nn.utils.rnn.PackedSequence)
        # X is a PACKED sequence, there is no need to permute
        
        rnn_out, self.hidden = self.basic_rnn(X)
        if isinstance(self.basic_rnn, nn.LSTM):
            self.hidden, self.cell = self.hidden
        
        if is_packed:
            # unpack the output
            batch_first_output, seq_sizes = rnn_utils.pad_packed_sequence(rnn_out, batch_first=True)        
            seq_slice = torch.arange(seq_sizes.size(0))
        else:
            batch_first_output = rnn_out
            seq_sizes = 0 # so it is -1 as the last output
            seq_slice = slice(None, None, None) # same as ':'
            
        # Последний выход: (N, H)
        last_output = batch_first_output[seq_slice, seq_sizes-1]
        
        # Логиты: (N, n_outputs)
        out = self.classifier(last_output)
                
        # final output is (N, n_outputs)
        return out.view(-1, self.n_outputs)

Авторская модель выбирает последний действительный **output**, включая его обратную часть. Это допустимый readout, но не конкатенация двух конечных состояний: обратное конечное состояние находится на первом шаге. Сохраняем пример Годоя и не называем этот вектор двумя конечными состояниями.

### Конфигурация и обучение

In [ ]:
torch.manual_seed(21)
model = SquareModelOne(n_features=2, hidden_dim=2, n_outputs=1,
                       rnn_layer=nn.LSTM, num_layers=1, bidirectional=True)
loss = nn.BCEWithLogitsLoss()
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
history_one = fit_sequence_model(model, loss, optimizer, train_var_loader, None, n_epochs=100)
model_one = model

In [ ]:
metrics_one = evaluate_sequence_model(model_one, train_var_loader, loss, torch.device('cpu'))
metrics_one

<a id="section-17"></a>

## 17. От последовательностей к CRNN

В классификации мы превращали карту признаков в один вектор. Для чтения строки
сохраним порядок слева направо: один столбец карты станет одним входным вектором.

**Вопрос:** что потеряется, если усреднить карту по ширине?
Почему один столбец признаков не обязан соответствовать одному символу?

In [ ]:
features_demo = torch.arange(4 * 6).reshape(1, 4, 1, 6).float()
sequence_demo = features_demo.squeeze(2).permute(2, 0, 1)
print('Карта (N, C, H, W):', tuple(features_demo.shape))
print('Последовательность (T, N, C):', tuple(sequence_demo.shape))
print('Первый вектор:', sequence_demo[0, 0])
seminar_plots.plot_feature_sequence(features_demo);

### T1. Карта и последовательность

Карта имеет форму `(2, 8, 1, 5)`. Получите `(5, 2, 8)` так, чтобы шаг t содержал столбец t. Сколько здесь шагов и признаков?

In [ ]:
task_features = torch.arange(2 * 8 * 5).reshape(2, 8, 1, 5)
# Ваш код здесь

<details>
<summary>Решение и проверка</summary>

```python
task_sequence = task_features.squeeze(2).permute(2, 0, 1)
assert task_sequence.shape == (5, 2, 8)
torch.testing.assert_close(task_sequence[3, 1], task_features[1, :, 0, 3])
```

</details>

### Собираем CRNN

Прикладной пример курса: уже вырезанное изображение номера, не поиск номера
на фотографии. Сохраняем размер `(3, 32, 160)` и архитектуру прошлого года.

Сначала построим только CNN. Её выход: `(N, 512, 1, 20)`.
Высоту убираем, ширину превращаем в шаги: `(20, N, 512)`.

**Вопрос:** почему нельзя заменить перестановку осей произвольным `reshape`?

In [ ]:
class Encoder(nn.Module):
    def __init__(self, imgH):
        super(Encoder, self).__init__()
        assert imgH % 16 == 0, 'imgH has to be a multiple of 16'
        self.cnn = nn.Sequential(
         nn.Conv2d(   3,  64,     3, 1, 1), nn.PReLU(), nn.MaxPool2d((2,2)),               # 64x16x80
         nn.Conv2d(  64, 128,     3, 1, 1), nn.PReLU(), nn.MaxPool2d((2,2)),               # 128x8x40
         nn.Conv2d( 128, 256,     3, 1, 1), nn.BatchNorm2d(256), nn.PReLU(),               # 256x8x40
         nn.Conv2d( 256, 256,     3, 1, 1), nn.PReLU(), nn.MaxPool2d((2,2), (2,2), (0,0)), # 256x4x20
         nn.Conv2d( 256, 512,     3, 1, 1), nn.BatchNorm2d(512), nn.PReLU(),               # 512x4x20
         nn.Conv2d( 512, 512,     3, 1, 1), nn.PReLU(), nn.MaxPool2d((2,1), (2,1), (0,0)), # 512x2x20
         nn.Conv2d( 512, 512, (2,1), 1, 0), nn.BatchNorm2d(512), nn.PReLU()                # 512x1x20
        )
    def forward(self, input):
        conv = self.cnn( input )
        b, c, h, w = conv.size()
        assert h == 1, "Ожидается высота карты 1"
        conv   = conv.squeeze(2      )
        output = conv.permute(2, 0, 1)  # [w, b, c]

        return output

In [ ]:
encoder = Encoder(32).to(device).eval()
synthetic_images = torch.rand(2, 3, 32, 160, device=device)
with torch.no_grad():
    conv = encoder.cnn(synthetic_images)
    squeezed = conv.squeeze(2)
    sequence = squeezed.permute(2, 0, 1)
    torch.testing.assert_close(sequence, encoder(synthetic_images))
print('После CNN:', tuple(conv.shape))
print('Без высоты:', tuple(squeezed.shape))
print('Для LSTM:', tuple(sequence.shape))
seminar_plots.plot_feature_sequence(conv[:, :8]);

### Рекуррентный блок и посимвольная проекция

Оставляем имя `Decoder` из исходника, но это **не авторегрессионный декодер**:
он не получает предыдущий предсказанный символ. BiLSTM обогащает признаки
контекстом, `Linear` выдаёт логиты символов, включая blank.
Строку мы получим позже отдельным алгоритмом декодирования.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, nIn, nHidden, nL, nClass):
        super(Decoder, self).__init__()
        self.rnn  = nn.LSTM(nIn, nHidden, num_layers=nL, bidirectional=True)
        self.proj = nn.Linear(2*nHidden, nClass)
    def forward(self, input):

        y, _ = self.rnn(input)

        T, b, h      = y.size()
        t_rec        = y.view(T * b, h)

        output = self.proj(t_rec)  # [T * b, nOut]
        output = output.view(T, b, -1)

        return output

In [ ]:
class ConvolutionalRecurrentNeuralNetwork(nn.Module):
    def __init__(self, imgH, nIn, nH, nL, nClass):
        super().__init__()
        self.encoder = Encoder(imgH)
        self.decoder = Decoder(nIn, nH, nL, nClass)

    def forward(self, images):
        sequence = self.encoder(images)
        logits = self.decoder(sequence)
        return logits

abc = '0123456789ABEKMHOPCTYX'
imgH, nIn, nH, nL, nClass = 32, 512, 512, 1, len(abc) + 1
torch.manual_seed(42)
crnn = ConvolutionalRecurrentNeuralNetwork(imgH, nIn, nH, nL, nClass).to(device)
crnn.eval()
with torch.no_grad():
    logits = crnn(synthetic_images)
assert logits.shape == (20, 2, 23)
print('Логиты (T, N, C):', tuple(logits.shape))
print('Обучаемых параметров:', sum(p.numel() for p in crnn.parameters()))

<a id="section-18"></a>

## 18. CTC и декодирование

Известна строка, но неизвестно, какой шаг отвечает за каждый её символ.
Путь $\pi$ задаёт символ или blank на каждом шаге. Операция $B$ сначала
объединяет соседние одинаковые метки, затем удаляет blank.

$$p(y\mid x)=\sum_{\pi:B(\pi)=y}\prod_{t=1}^{T}p_t(\pi_t\mid x),
\qquad \ell(x,y)=-\log p(y\mid x).$$

CTC учитывает **все допустимые выравнивания** целевой строки, а не один
максимально вероятный путь. В реализации сумма вычисляется динамическим
программированием; полный перебор ниже нужен только для маленького примера.

Blank не является пробелом или padding. Он не входит в целевую строку.
**Вопрос:** чем отличаются пути `A A blank` и `A blank A`?

In [ ]:
def collapse_path(path, blank=0):
    result = []
    previous = None
    for label in path:
        if label != previous and label != blank:
            result.append(label)
        previous = label
    return tuple(result)

print('A A blank:', collapse_path([1, 1, 0]))
print('A blank A:', collapse_path([1, 0, 1]))

### T3. Повторы и blank

Предскажите результат collapse для каждого пути. Почему нельзя сначала удалить все blank?

In [ ]:
task_paths = [(1, 1, 0, 1), (0, 1, 1, 0), (2, 0, 2, 2)]
# Ваш код здесь

<details>
<summary>Решение и проверка</summary>

```python
task_results = [collapse_path(p) for p in task_paths]
assert task_results == [(1, 1), (1,), (2, 2)]
assert collapse_path([1, 0, 1]) != collapse_path([1, 1])
```

</details>

In [ ]:
from itertools import product

# Полный перебор и проверка производной: float64 только на CPU.
toy_logits = torch.tensor([[0.2, 0.6, -0.3], [0.7, 0.1, -0.2],
                           [0.1, 0.5, 0.4]], dtype=torch.float64,
                          requires_grad=True)
toy_probs = toy_logits.softmax(-1)
target = (1, 1)
paths = [p for p in product(range(3), repeat=3) if collapse_path(p) == target]
probability = sum(torch.stack([toy_probs[t, label] for t, label in enumerate(p)]).prod()
                  for p in paths)
enumerated_loss = -probability.log()
library_loss = F.ctc_loss(toy_logits.log_softmax(-1).unsqueeze(1),
                         torch.tensor(target), torch.tensor([3]),
                         torch.tensor([2]), reduction='sum')
torch.testing.assert_close(enumerated_loss, library_loss)
g_enum = torch.autograd.grad(enumerated_loss, toy_logits, retain_graph=True)[0]
g_ctc = torch.autograd.grad(library_loss, toy_logits)[0]
torch.testing.assert_close(g_enum, g_ctc)
print('Пути для AA:', paths)
print('Потеря перебора и PyTorch:', enumerated_loss.item(), library_loss.item())
seminar_plots.plot_ctc(toy_probs.detach(), ['blank', 'A', 'B']);

Для `AA` на трёх шагах допустим только `A blank A`. Теперь заменим цель на `A`:
допустимых путей станет несколько. **Вопрос:** что изменится, если вместо суммы
их вероятностей взять только максимальную?

In [ ]:
target_a = (1,)
paths_a = [p for p in product(range(3), repeat=3) if collapse_path(p) == target_a]
probs_a = torch.stack([torch.stack([toy_probs.detach()[t, v]
                                  for t, v in enumerate(p)]).prod() for p in paths_a])
sum_loss = -probs_a.sum().log()
best_path_loss = -probs_a.max().log()
expected = F.ctc_loss(toy_logits.detach().log_softmax(-1).unsqueeze(1),
                      torch.tensor(target_a), torch.tensor([3]),
                      torch.tensor([1]), reduction='sum')
torch.testing.assert_close(sum_loss, expected)
print('Путей для A:', len(paths_a))
print('CTC:', sum_loss.item(), 'только лучший путь:', best_path_loss.item())

### Длины и нормировка

Для цели длины $U$ с $r$ соседними одинаковыми парами нужно хотя бы $U+r$
шагов. Для `AA` двух шагов недостаточно. `zero_infinity=True` обнулит бесконечную
потерю и её градиент, но не исправит невозможное выравнивание.

При `reduction='mean'` PyTorch сначала делит потерю каждого объекта на длину
его цели, затем усредняет по батчу. Это не просто среднее ненормированных потерь.

In [ ]:
short_log_probs = torch.zeros(2, 1, 3).log_softmax(-1)
args = (short_log_probs, torch.tensor([1, 1]), torch.tensor([2]), torch.tensor([2]))
invalid_loss = F.ctc_loss(*args, reduction='sum')
masked_loss = F.ctc_loss(*args, reduction='sum', zero_infinity=True)
assert torch.isinf(invalid_loss) and masked_loss == 0
print('Невозможное выравнивание:', invalid_loss.item())
print('С zero_infinity:', masked_loss.item(), '(пример пропущен, а не исправлен)')

### Greedy-декодирование

Выберем наиболее вероятную метку на каждом шаге и применим collapse.
Это best-path decoding. Наиболее вероятный путь не обязательно даёт
наиболее вероятную строку: у другой строки может быть больше суммарная вероятность.
Алгоритм Витерби для общего HMM не является реализацией CTC-loss.

In [ ]:
def decode(logits, alphabet=abc):
    paths = logits.argmax(dim=-1).transpose(0, 1).detach().cpu().tolist()
    return [''.join(alphabet[label - 1] for label in collapse_path(path))
            for path in paths]

# blank вероятнее A на каждом шаге, но строка A вероятнее пустой строки.
ambiguous = torch.tensor([[0.6, 0.4], [0.6, 0.4]])
mass = {}
for path in product(range(2), repeat=2):
    text = collapse_path(path)
    mass[text] = mass.get(text, 0.0) + float(ambiguous[0, path[0]] * ambiguous[1, path[1]])
print('Greedy:', decode(ambiguous.log().unsqueeze(1), 'A'))
print('Вероятности строк:', mass)
assert mass[(1,)] > mass[()]

<a id="section-19"></a>

## 19. Обучение OCR и проверка распознаваний

Длины текстов могут различаться. Склеим числовые цели и отдельно сохраним длины.
Индекс 0 оставляем для blank; буква с индексом i в алфавите получает код i+1.
На вход CTCLoss подаём `log_softmax`, а не результат `argmax`.

**Вопрос:** через какую из этих операций проходит градиент к CNN?

In [ ]:
def encode_text(text, alphabet=abc):
    return torch.tensor([alphabet.index(char) + 1 for char in text], dtype=torch.long)

def collate_fn(batch):
    sequences = [encode_text(item['text']) for item in batch]
    return {
        'images': torch.stack([item['image'] for item in batch]),
        'targets': torch.cat(sequences),
        'target_lengths': torch.tensor([len(s) for s in sequences], dtype=torch.long),
        'texts': [item['text'] for item in batch],
    }

synthetic_batch = collate_fn([
    {'image': synthetic_images[0].cpu(), 'text': 'A123BC77'},
    {'image': synthetic_images[1].cpu(), 'text': 'A123BC777'},
])
print('Цели:', synthetic_batch['targets'].shape,
      'длины:', synthetic_batch['target_lengths'].tolist())

### T4. Батч целей

Подготовьте склеенные targets и target_lengths для двух строк. Не добавляйте blank между символами разметки.

In [ ]:
task_texts = ['A123BC77', 'A123BC777']
# Ваш код здесь

<details>
<summary>Решение и проверка</summary>

```python
task_sequences = [encode_text(text) for text in task_texts]
task_targets = torch.cat(task_sequences)
task_lengths = torch.tensor([len(s) for s in task_sequences], dtype=torch.long)
assert task_lengths.tolist() == [8, 9]
assert task_targets.numel() == task_lengths.sum() == 17
assert (task_targets > 0).all()
```

</details>

In [ ]:
def train_step(model, batch, optimizer, device):
    model.train()
    images = batch['images'].to(device)
    targets = batch['targets'].to(device)
    lengths = batch['target_lengths']
    optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    input_lengths = torch.full((images.size(0),), logits.size(0), dtype=torch.long)
    loss = F.ctc_loss(logits.log_softmax(-1), targets, input_lengths, lengths,
                      blank=0, reduction='mean', zero_infinity=False)
    if not torch.isfinite(loss):
        raise ValueError('Неконечная потеря: проверьте длины и допустимые выравнивания.')
    loss.backward()
    if not all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()):
        raise ValueError('Проверьте градиенты модели.')
    optimizer.step()
    return loss.detach().item()

optimizer = torch.optim.Adam(crnn.parameters(), lr=3e-4, amsgrad=True, weight_decay=1e-4)
step_loss = train_step(crnn, synthetic_batch, optimizer, device)
print('Проверка одного шага на случайных изображениях:', step_loss)
print('Это проверка вычислений, не обучение распознаванию номеров.')

### Настоящие данные: архив и фиксированное разбиение

Используем исходный `seminar_crnn_data.zip`: [постоянная ссылка версии v1](https://github.com/SergeyMalashenko/2026_ML3/releases/download/seminar08-data-v1/seminar_crnn_data.zip).
Загрузка не требует Google Drive или входа в аккаунт. Архив проверяется по SHA256;
распаковка и веса находятся в пользовательском кеше, вне репозитория.

В разметке 41 141 изображение. Как в прошлом году, оставляем номера формата
«буква, три цифры, две буквы, две или три цифры региона» из алфавита `abc`.
Получаем 31 345 примеров. В фиксированном split: train 25 077, validation 6 268;
идентичные RGB-изображения не пересекают границу выборок. Проверено чтение всех
размеченных изображений. Это защита от точных дубликатов, не гарантия отсутствия
похожих кадров одного автомобиля. Validation используется для выбора весов,
а не объявляется независимым финальным тестом.

**Вопрос:** почему случайного разбиения по именам файлов недостаточно при наличии копий одного изображения?

In [ ]:
import hashlib
import json
import shutil
import stat
import tempfile
from pathlib import PurePosixPath
from urllib.request import urlopen
from zipfile import ZipFile

OCR_RELEASE = 'https://github.com/SergeyMalashenko/2026_ML3/releases/download/seminar08-data-v1'
OCR_ARCHIVE_SHA256 = '2e613f3fcdda10a0f5078f5219cae5f8f53fe40599995f1eb9cdffc612ab6d68'
OCR_SPLIT_SHA256 = 'b1147a04f753e062df00adf34aefff5ebe62fd49dfe554f0b7c5cece784a72f6'
ocr_cache = Path(os.environ.get('ML3_OCR_CACHE', Path.home()/'.cache/ai-masters/ml3/seminar08'))
ocr_cache.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_verified(filename, expected_sha256, local_source=None):
    destination = ocr_cache/filename
    if destination.is_file() and file_sha256(destination) == expected_sha256:
        return destination
    with tempfile.NamedTemporaryFile(dir=ocr_cache, delete=False) as stream:
        temporary = Path(stream.name)
        try:
            if local_source is not None:
                with Path(local_source).open('rb') as source:
                    shutil.copyfileobj(source, stream)
            else:
                with urlopen(f'{OCR_RELEASE}/{filename}', timeout=60) as source:
                    shutil.copyfileobj(source, stream)
            stream.flush()
            if file_sha256(temporary) != expected_sha256:
                raise ValueError(f'Контрольная сумма не совпала: {filename}')
            stream.close()
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
    return destination

archive_path = fetch_verified('seminar_crnn_data.zip', OCR_ARCHIVE_SHA256,
                              os.environ.get('ML3_OCR_ARCHIVE'))
split_path = fetch_verified('ocr-split-v1.json', OCR_SPLIT_SHA256)
ocr_data_root = ocr_cache/'data-v1'
marker = ocr_data_root/'.archive-sha256'
if not marker.is_file() or marker.read_text().strip() != OCR_ARCHIVE_SHA256:
    if ocr_data_root.exists():
        raise RuntimeError('Непроверенная папка data-v1: выберите другой ML3_OCR_CACHE.')
    with tempfile.TemporaryDirectory(dir=ocr_cache) as directory:
        staging = Path(directory)/'data'
        staging.mkdir()
        with ZipFile(archive_path) as archive:
            for item in archive.infolist():
                path = PurePosixPath(item.filename)
                if (path.is_absolute() or '..' in path.parts or '\\' in item.filename
                        or stat.S_ISLNK(item.external_attr >> 16)):
                    raise ValueError('Недопустимый путь в архиве')
            archive.extractall(staging)
        (staging/'.archive-sha256').write_text(OCR_ARCHIVE_SHA256)
        staging.rename(ocr_data_root)
ocr_split = json.loads(split_path.read_text())
assert ocr_split['archive_sha256'] == OCR_ARCHIVE_SHA256
assert not ({r['image_sha256'] for r in ocr_split['train']} &
            {r['image_sha256'] for r in ocr_split['validation']})
print('Train:', len(ocr_split['train']), 'Validation:', len(ocr_split['validation']))

### От файла к батчу

RGB, размер 32×160 и нормировка сохранены из прошлогоднего семинара.
`Dataset` возвращает изображение и строку; уже написанный `collate_fn` собирает
тензоры и склеенные CTC-цели. Число шагов выхода CRNN равно 20.

**Вопрос:** сколько дополнительных шагов нужно для строки с соседними одинаковыми символами?

In [ ]:
from PIL import Image
from torchvision.transforms.v2 import Compose, ToImage, ToDtype, Resize, Normalize

ocr_transform = Compose([
    ToImage(), ToDtype(torch.float32, scale=True), Resize((32, 160), antialias=True),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class RecognitionDataset(torch.utils.data.Dataset):
    def __init__(self, root, records):
        self.root, self.records = Path(root), records
        for row in records:
            text = row['text']
            assert text and all(char in abc for char in text)
            assert len(text)+sum(a==b for a,b in zip(text,text[1:])) <= 20
            assert PurePosixPath(row['file']).name == row['file']
            assert (self.root/'images'/row['file']).is_file()

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(self.root/'images'/row['file']) as image:
            tensor = ocr_transform(image.convert('RGB'))
        return {'image': tensor, 'text': row['text']}

ocr_train_dataset = RecognitionDataset(ocr_data_root, ocr_split['train'])
ocr_val_dataset = RecognitionDataset(ocr_data_root, ocr_split['validation'])
ocr_train_loader = torch.utils.data.DataLoader(
    ocr_train_dataset, batch_size=128, shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(42), collate_fn=collate_fn)
ocr_val_loader = torch.utils.data.DataLoader(
    ocr_val_dataset, batch_size=128, shuffle=False, num_workers=0, collate_fn=collate_fn)
real_batch = collate_fn([ocr_train_dataset[i] for i in range(4)])
print(real_batch['images'].shape, real_batch['target_lengths'].tolist())
seminar_plots.plot_ocr_samples(real_batch['images'], real_batch['texts']);

### Ошибка по символам и по целой строке

Greedy-декодирование применяет argmax по классам и CTC-collapse. Для качества
считаем долю полностью совпавших строк и CER: суммарное расстояние Левенштейна,
делённое на суммарное число символов в разметке. CER может быть больше единицы.
Loss усредняется по объектам; `CTCLoss(reduction='mean')` дополнительно нормирует
потерю каждого объекта на длину его цели.

**Вопрос:** если в девятисимвольном номере неверна одна буква, сколько даст exact match?

In [ ]:
def decode_ocr(logits):
    paths = logits.detach().argmax(-1).transpose(0, 1).cpu().tolist()
    return [''.join(abc[k-1] for k in collapse_path(path)) for path in paths]

def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis)+1))
    for i, expected in enumerate(reference, 1):
        current = [i]
        for j, predicted in enumerate(hypothesis, 1):
            current.append(min(current[-1]+1, previous[j]+1,
                               previous[j-1]+(expected != predicted)))
        previous = current
    return previous[-1]

def evaluate_ocr(model, loader, target_device):
    model.eval()
    total_loss = exact = edits = characters = count = 0
    predictions = []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['images'].to(target_device))
            lengths = torch.full((len(batch['texts']),), logits.size(0), dtype=torch.long)
            loss = F.ctc_loss(logits.log_softmax(-1), batch['targets'].to(target_device),
                              lengths, batch['target_lengths'], blank=0, zero_infinity=False)
            if not torch.isfinite(loss):
                raise ValueError('Неконечная validation loss')
            decoded = decode_ocr(logits)
            for expected, predicted in zip(batch['texts'], decoded):
                exact += expected == predicted
                edits += edit_distance(expected, predicted)
                characters += len(expected)
            predictions.extend(decoded)
            total_loss += loss.item()*len(decoded)
            count += len(decoded)
    return {'loss':total_loss/count, 'exact_match':exact/count,
            'cer':edits/characters, 'count':count}, predictions

assert edit_distance('A123BC77', 'A123BC77') == 0
assert edit_distance('A123BC77', 'A123BC7') == 1
assert edit_distance('A123BC77', '') == 8

### Короткий учебный запуск

Создаём новую модель, отдельно от синтетической проверки и от будущих заранее
обученных весов. Два шага на настоящих изображениях показывают работу цикла,
но не обещают качественного распознавания.

In [ ]:
torch.manual_seed(42)
ocr_demo = ConvolutionalRecurrentNeuralNetwork(imgH, nIn, nH, nL, nClass).to(device)
ocr_demo_optimizer = torch.optim.Adam(ocr_demo.parameters(), lr=3e-4,
                                      amsgrad=True, weight_decay=1e-4)
demo_loader = torch.utils.data.DataLoader(ocr_train_dataset, batch_size=8,
    shuffle=False, num_workers=0, collate_fn=collate_fn)
for step, batch in enumerate(demo_loader):
    if step == 2:
        break
    print('Учебный шаг', step+1, 'loss:', train_step(ocr_demo, batch, ocr_demo_optimizer, device))
print('Это новая модель после двух шагов, не заранее обученный checkpoint.')

### Полный цикл для самостоятельного запуска

По умолчанию длительное обучение выключено. Для воспроизведения задайте
`OCR_TRAIN_EPOCHS=10`. Это отдельный эксперимент с новой инициализацией;
checkpoint выбирается по минимальной validation loss. Логи и веса сохраняются
в кеше, а не в репозитории. После него ниже явно загружается опубликованный
учебный checkpoint, а не молча используются веса текущего эксперимента.

In [ ]:
OCR_TRAIN_EPOCHS = 0
if OCR_TRAIN_EPOCHS:
    torch.manual_seed(42)
    ocr_experiment = ConvolutionalRecurrentNeuralNetwork(imgH, nIn, nH, nL, nClass).to(device)
    ocr_optimizer = torch.optim.Adam(ocr_experiment.parameters(), lr=3e-4,
                                   amsgrad=True, weight_decay=1e-4)
    loader = torch.utils.data.DataLoader(ocr_train_dataset, batch_size=128, shuffle=True,
        num_workers=0, generator=torch.Generator().manual_seed(42), collate_fn=collate_fn)
    best_loss, experiment_history = float('inf'), []
    for epoch in range(OCR_TRAIN_EPOCHS):
        total, count = 0.0, 0
        for batch in loader:
            loss_value = train_step(ocr_experiment, batch, ocr_optimizer, device)
            total += loss_value*len(batch['texts'])
            count += len(batch['texts'])
        metrics, _ = evaluate_ocr(ocr_experiment, ocr_val_loader, device)
        experiment_history.append({'epoch':epoch+1, 'train_loss':total/count, **metrics})
        print(experiment_history[-1])
        if metrics['loss'] < best_loss:
            best_loss = metrics['loss']
            torch.save({'model_state':{k:v.detach().cpu() for k,v in ocr_experiment.state_dict().items()},
                        'epoch':epoch+1, 'metrics':metrics, 'alphabet':abc,
                        'archive_sha256':OCR_ARCHIVE_SHA256, 'split_sha256':OCR_SPLIT_SHA256},
                       ocr_cache/'my-crnn-best.pt')
        (ocr_cache/'my-training-history.json').write_text(json.dumps(experiment_history, indent=2))
else:
    print('Полное обучение выключено: OCR_TRAIN_EPOCHS = 0.')

### Заранее обученные веса: отдельная модель

Здесь загружается самостоятельный checkpoint с зафиксированным протоколом.
Он не является результатом двух учебных шагов выше. Перед загрузкой проверяются
SHA256, алфавит и версия разбиения. Архитектура сохраняется исходной.

[Протокол обучения](https://github.com/SergeyMalashenko/2026_ML3/releases/download/seminar08-data-v1/training-protocol.json):
10 эпох на MPS, Adam, batch size 128. По минимальной validation loss выбрана
эпоха 10: exact match 98,53%, CER 0,23% на 6 268 примерах. Это результат одного
запуска и выбора checkpoint на validation, а не независимая тестовая оценка.

In [ ]:
CHECKPOINT_SHA256 = '95d286de3ce4a8903963870d8f92d44602443196f54a0c8025dfaaed35f5834e'
checkpoint_ready = CHECKPOINT_SHA256 is not None
if checkpoint_ready:
    checkpoint_path = fetch_verified('crnn-baseline-v1.pt', CHECKPOINT_SHA256)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    assert checkpoint['alphabet'] == abc
    assert checkpoint['archive_sha256'] == OCR_ARCHIVE_SHA256
    assert checkpoint['split_sha256'] == OCR_SPLIT_SHA256
    ocr_model = ConvolutionalRecurrentNeuralNetwork(imgH, nIn, nH, nL, nClass).to(device)
    ocr_model.load_state_dict(checkpoint['model_state'])
    ocr_model.eval()
    print('Загружен отдельный checkpoint. Эпоха:', checkpoint['epoch'])
    print('Метрики протокола:', checkpoint['metrics'])

In [ ]:
if checkpoint_ready:
    preview_loader = torch.utils.data.DataLoader(ocr_val_dataset, batch_size=8,
        shuffle=False, num_workers=0, collate_fn=collate_fn)
    preview_batch = next(iter(preview_loader))
    with torch.no_grad():
        preview_logits = ocr_model(preview_batch['images'].to(device))
    preview_predictions = decode_ocr(preview_logits)
    seminar_plots.plot_ocr_samples(preview_batch['images'], preview_batch['texts'], preview_predictions)
    print('Первые 8 примеров validation в фиксированном порядке, без отбора удачных.')

### Повторная оценка и разбор ошибок

Полная validation-оценка может занять несколько минут; её можно включить отдельно.
Опубликованный протокол относится ко всем 6 268 примерам, а не к показанным восьми.
**Вопрос:** почему хороших отдельных примеров недостаточно, чтобы оценить модель?

In [ ]:
OCR_FULL_VALIDATION = False
if checkpoint_ready and OCR_FULL_VALIDATION:
    current_metrics, predictions = evaluate_ocr(ocr_model, ocr_val_loader, device)
    print('Повторная оценка на всех примерах:', current_metrics)
    mistakes = [i for i, (row, predicted) in enumerate(zip(ocr_split['validation'], predictions))
                if row['text'] != predicted][:8]
    if mistakes:
        error_batch = collate_fn([ocr_val_dataset[i] for i in mistakes])
        seminar_plots.plot_ocr_samples(error_batch['images'], error_batch['texts'],
                                      [predictions[i] for i in mistakes])
        print('Первые ошибки в порядке validation, не случайная выборка.')
elif checkpoint_ready:
    print('Полная validation-оценка выключена; выше приведены метрики сохранённого протокола.')

In [ ]:
ocr_ready = True
print('Реальные данные подключены. Заранее обученные веса загружены:', checkpoint_ready)

<a id="section-20"></a>

## 20. Итоги и следующий шаг

CNN извлекает пространственные признаки. RNN обрабатывает их как последовательность.
CTC позволяет учиться по строке без разметки каждого шага. Greedy-decoder превращает
пошаговые предсказания в текст, но не является функцией потерь.

**Проверка понимания:** где находятся обучаемые параметры? Где меняется только
форма? Зачем blank? Почему `argmax` не находится между моделью и loss?

CTC задаёт монотонное выравнивание и факторизует вероятность пути по шагам
при заданном входе. Это не означает, что CNN/BiLSTM не используют контекст.
Далее можно сравнить этот подход с encoder–decoder и attention: там другой
способ получения выходной последовательности. Сегодня мы его не реализовали.

<a id="stn"></a>

## Приложение. STN

Отдельное расширение базовой модели. Сеть локализации предсказывает аффинное преобразование;
`affine_grid` строит координаты, `grid_sample` получает преобразованное изображение.
Начальное преобразование должно быть единичным. Обе операции используют
одинаковый `align_corners=False`. Это необходимая поправка к прошлогоднему коду.

**Вопрос:** почему сеть локализации получает градиент, хотя целевых углов поворота
или сдвигов в разметке нет? Улучшение OCR здесь пока не проверено.

In [ ]:
class STN(nn.Module):
    def __init__(self):
        super(STN, self).__init__()

        self.localization = nn.Sequential(
          #  loc conv1
          nn.Conv2d( 3, 32, kernel_size=5, stride=1, padding=2, bias=True ),
          nn.MaxPool2d( 2, stride=2 ),
          nn.PReLU(),
          #  loc conv2
          nn.Conv2d( 32, 32, kernel_size=3, stride=1, padding=2, bias=True ),
          nn.MaxPool2d( 2, stride=2 ),
          nn.PReLU(),
          #  loc conv3
          nn.Conv2d( 32, 64, kernel_size=3, stride=1, padding=2, bias=True ),
          nn.MaxPool2d( 2, stride=2 ),
          nn.PReLU()
        )
        self.f_size = 105

        self.fc_loc = nn.Sequential(
          nn.Linear( 64 * self.f_size, 64, bias=True ),
          nn.PReLU(),
          nn.Linear( 64, 3 * 2, bias=True )
        )

        # initial weights
        with torch.no_grad():
            self.fc_loc[2].weight.zero_()
            self.fc_loc[2].bias.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float32))

    def forward(self, x):
        xs = self.localization( x )
        xs = xs.view( -1, 64 *  self.f_size )

        theta = self.fc_loc( xs )
        theta = theta.view( -1, 2, 3 )

        grid  = F.affine_grid( theta, x.size(), align_corners=False )
        x = F.grid_sample(x, grid, align_corners=False)
        return x

In [ ]:
stn = STN().to(device)
stn_input = synthetic_images.detach().clone().requires_grad_()
stn_output = stn(stn_input)
torch.testing.assert_close(stn_output, stn_input, atol=5e-5, rtol=1e-4)
stn_output.square().mean().backward()
assert torch.isfinite(stn_input.grad).all()
assert torch.isfinite(stn.fc_loc[2].weight.grad).all()
print('Начальное преобразование единичное; градиенты конечны.')
seminar_plots.plot_stn(stn_input, stn_output);

In [ ]:
class ConvolutionalRecurrentNeuralNetworkSTN(ConvolutionalRecurrentNeuralNetwork):
    def __init__(self, imgH, nIn, nH, nL, nClass):
        super().__init__(imgH, nIn, nH, nL, nClass)
        self.spatial_transformer = STN()

    def forward(self, images):
        return super().forward(self.spatial_transformer(images))

crnn_stn = ConvolutionalRecurrentNeuralNetworkSTN(imgH, nIn, nH, nL, nClass).to(device)
lr = 3e-4
parameter_s = [
    {'params': crnn_stn.spatial_transformer.parameters(), 'lr': lr},
    {'params': crnn_stn.encoder.parameters(), 'lr': lr * 0.001},
    {'params': crnn_stn.decoder.parameters(), 'lr': lr * 0.001},
]
optimizer_stn = torch.optim.Adam(parameter_s, amsgrad=True, weight_decay=1e-4)
print('Learning rate групп:', [g['lr'] for g in optimizer_stn.param_groups])
if checkpoint_ready:
    crnn_stn.encoder.load_state_dict(ocr_model.encoder.state_dict())
    crnn_stn.decoder.load_state_dict(ocr_model.decoder.state_dict())
print('Для STN нужны отдельное обучение и сравнение; улучшение качества не заявляется.')

## Источники

- D. V. Godoy: [Chapter 8](https://github.com/dvgodoy/PyTorchStepByStep/blob/master/Chapter08.ipynb).
- [Прошлогодний CRNN](https://github.com/SergeyMalashenko/2025_ML3/blob/main/seminars/09_crnn/seminar09_crnn.ipynb).
- [CRNN: Shi et al.](https://arxiv.org/abs/1507.05717).
- [CTC: Graves et al.](https://www.cs.toronto.edu/~graves/icml_2006.pdf).
- [PyTorch CTCLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CTCLoss.html).
- [Spatial Transformer Networks](https://arxiv.org/abs/1506.02025).

Генератор последовательностей Годоя сохранён; перевод, подбор учебных шагов,
CTC-примеры и задания относятся к адаптации курса. Лицензия: `LICENSE-GODOY`.